# TP - Clasificación Multiclase de Texto con Regresión Softmax
### Redes Neuronales / Deep Learning

Dataset: **20 Newsgroups** (~18.000 posts de foros de Usenet, clasificados en 20 categorías).

## Objetivos
- Implementar un pipeline de preprocesamiento de texto **configurable** (stopwords, stemming, lematización, n-gramas).
- Implementar una **Regresión Softmax** (regresión logística multinomial) en PyTorch: una única capa lineal + `CrossEntropyLoss`, sin capas ocultas.
- Trackear el entrenamiento con **TensorBoard** y **MLflow**.
- Usar **Optuna** para buscar la mejor combinación de hiperparámetros de preprocesamiento *y* de entrenamiento en conjunto.

## Qué hay que completar
Los bloques marcados `# TODO` son los que tienen que completar ustedes. El resto ya está resuelto.

Esta notebook es individual/standalone: no tiene una sección de defensa grupal asociada.


## 0. Setup

### Herramientas que uso y para qué

- **PyTorch**: el framework con el que armo y entreno el modelo. Uso `nn.Linear` (la capa del modelo), `nn.CrossEntropyLoss` (la función de pérdida), `optim.Adam` (el optimizador que actualiza los pesos) y `DataLoader` (reparte los datos en mini-batches y los mezcla en cada época). Lo más importante es que calcula solo los gradientes con `loss.backward()`, así no tengo que derivar a mano.
- **scikit-learn**: todo lo que no es la red: bajar el dataset, pasar el texto a números con `TfidfVectorizer`, separar train/validación con `train_test_split`, calcular métricas (accuracy, matriz de confusión, reporte por clase) y los baselines de comparación.
- **NLTK**: herramientas de lenguaje: tokenizar (cortar el texto en palabras), la lista de stopwords en inglés, el stemmer de Porter y el lematizador de WordNet.
- **TensorBoard**: muestra las curvas de loss y accuracy época por época, de forma visual e interactiva.
- **MLflow**: lleva el registro de cada corrida: qué hiperparámetros usé, qué métricas obtuve en cada época y el modelo guardado. Sirve para comparar corridas y no perder nada.
- **Optuna**: busca automáticamente la mejor combinación de hiperparámetros. En vez de probar todo o probar a mano, usa los resultados de los trials anteriores para decidir qué probar después.
- **pandas / matplotlib / seaborn**: para armar las tablas y los gráficos del análisis.

In [ ]:
!pip install -q mlflow optuna


In [ ]:
import re
import time
import copy
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

from torch.utils.tensorboard import SummaryWriter
import mlflow
import mlflow.pytorch
import optuna

SEED = 42


def set_seed(seed):
    # Fijo todas las fuentes de azar (inicialización de pesos, mezcla de los batches, etc.)
    # para que una corrida se pueda repetir y dé lo mismo
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", device)

In [ ]:
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("punkt")
nltk.download("punkt_tab")


## 1. Datos: 20 Newsgroups

Usamos `remove=('headers', 'footers', 'quotes')` para sacar metadata que haría el problema "demasiado fácil" (por ejemplo, el header a veces incluye el nombre del newsgroup).

In [ ]:
newsgroups_train = fetch_20newsgroups(subset="train", remove=("headers", "footers", "quotes"))
newsgroups_test  = fetch_20newsgroups(subset="test",  remove=("headers", "footers", "quotes"))

class_names = newsgroups_train.target_names
NUM_CLASSES = len(class_names)
print(f"Clases ({NUM_CLASSES}):", class_names)
print(f"\nTrain: {len(newsgroups_train.data)} documentos | Test: {len(newsgroups_test.data)} documentos")

print("\n--- Ejemplo de documento ---")
print("Categoría:", class_names[newsgroups_train.target[0]])
print(newsgroups_train.data[0][:500])


In [ ]:
# Cantidad de documentos de cada clase en el set de entrenamiento
counts = np.bincount(newsgroups_train.target, minlength=NUM_CLASSES)
porcentajes = counts / counts.sum() * 100

# Cada porción se colorea según su grupo temático (el prefijo del nombre: comp, rec, sci, ...)
GROUP_COLORS = {
    "alt": "#2a78d6", "comp": "#eb6834", "misc": "#1baf7a", "rec": "#eda100",
    "sci": "#e87ba4", "soc": "#008300", "talk": "#4a3aa7",
}
colors = [GROUP_COLORS[name.split(".")[0]] for name in class_names]
labels = [f"{name}\n{c} docs ({p:.1f}%)" for name, c, p in zip(class_names, counts, porcentajes)]

fig, ax = plt.subplots(figsize=(12, 12))
ax.pie(
    counts,
    labels=labels,
    colors=colors,
    startangle=90,
    counterclock=False,
    labeldistance=1.06,
    wedgeprops={"edgecolor": "white", "linewidth": 2},
    textprops={"fontsize": 9, "color": "#333333"},
)
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in GROUP_COLORS.values()]
ax.legend(handles, GROUP_COLORS.keys(), title="Grupo", loc="upper left", bbox_to_anchor=(1.02, 1))
ax.set_title("Distribución de documentos por clase (train)", fontsize=14)
plt.tight_layout()
plt.show()


## NOTA

Como se puede ver la distribución de documentos en las diferentes clases del dataset de entrenamiento, estas estan muy bien balanceadas, por lo que no parece ser necesario realizar ningún balanceo de clases

## 2. Preprocesamiento configurable

Vamos a armar una función de preprocesamiento con estos hiperparámetros (todos van a formar parte de la búsqueda de Optuna más adelante):

- `remove_stopwords` (`True`/`False`): sacar palabras muy frecuentes y poco informativas (`the`, `is`, `and`, ...).
- `normalization` (`"none"` / `"stem"` / `"lemmatize"`):
  - `"none"`: no se normaliza la palabra.
  - `"stem"`: **stemming** con Porter Stemmer — corta la palabra a una raíz heurística (`"running"` → `"run"`, a veces resultados no son palabras reales: `"studies"` → `"studi"`).
  - `"lemmatize"`: **lematización** con WordNet — lleva la palabra a su forma de diccionario (`"running"` → `"run"`, `"studies"` → `"study"`), más lenta pero más "correcta" lingüísticamente.

### TODO 1: completar `preprocess_text`

Pasos esperados:
1. Tokenizar el texto con `word_tokenize`.
2. Pasar todo a minúsculas y quedarse solo con tokens alfabéticos (`token.isalpha()`), para descartar números y puntuación.
3. Si `remove_stopwords` es `True`, sacar los tokens que estén en el set de stopwords.
4. Según `normalization`, aplicar `stemmer.stem(token)`, `lemmatizer.lemmatize(token)`, o dejar el token igual.
5. Devolver los tokens unidos con espacios (un string, para poder pasárselo después a `TfidfVectorizer`).

In [ ]:
STOP_WORDS = set(stopwords.words("english"))
STEMMER = PorterStemmer()
LEMMATIZER = WordNetLemmatizer()


def preprocess_text(text, remove_stopwords, normalization):
    # Paso a minúsculas y corto el texto en tokens
    tokens = word_tokenize(text.lower())
    # Me quedo solo con tokens formados por letras (saco números y puntuación)
    tokens = [t for t in tokens if t.isalpha()]

    # Saco las stopwords ("the", "is", "and"...): aparecen en todos los temas y no ayudan a distinguirlos
    if remove_stopwords:
        tokens = [t for t in tokens if t not in STOP_WORDS]

    # Normalizo cada palabra: stemming corta a una raíz, lematización lleva a la forma de diccionario
    if normalization == "stem":
        tokens = [STEMMER.stem(t) for t in tokens]
    elif normalization == "lemmatize":
        tokens = [LEMMATIZER.lemmatize(t) for t in tokens]
    # si normalization == "none", la palabra queda igual

    # Devuelvo un string porque TfidfVectorizer espera texto
    return " ".join(tokens)

In [ ]:
# Sanity check: probamos las distintas combinaciones sobre una oración de ejemplo
ejemplo = "The researchers were studying the running processes and stopping conditions of the engines."
for norm in ["none", "stem", "lemmatize"]:
    for rm_sw in [False, True]:
        resultado = preprocess_text(ejemplo, remove_stopwords=rm_sw, normalization=norm)
        print(f"normalization={norm:10s} remove_stopwords={str(rm_sw):5s} -> {resultado}")


### Qué hace `preprocess_text`

Recibe un texto y lo devuelve "limpio", como un string de palabras separadas por espacios:
1. `word_tokenize(text.lower())`: paso todo a minúsculas (así "God" y "god" cuentan como la misma palabra) y corto el texto en tokens.
2. `t.isalpha()`: me quedo solo con los tokens que son letras. Se van números, signos de puntuación, direcciones de mail, etc.
3. Si `remove_stopwords` es `True`, saco las palabras que están en `STOP_WORDS`. Uso un `set` porque buscar una palabra ahí es instantáneo, aunque tenga cientos de palabras.
4. Según `normalization`: con `"stem"` corto cada palabra a su raíz (Porter), con `"lemmatize"` la llevo a su forma de diccionario (WordNet) y con `"none"` la dejo como está.
5. Uno los tokens con espacios, porque `TfidfVectorizer` recibe texto y no listas.

En el sanity check se ve la diferencia: el stemming convierte "studying" en "studi" y "conditions" en "condit" (raíces que no son palabras reales), mientras que la lematización deja "studying" igual (por defecto trata todo como sustantivo) y pasa "engines" a "engine". El stemming junta más formas de una misma palabra, aunque a veces mezcla palabras distintas.

## 3. Vectorización TF-IDF

Convertimos los textos preprocesados en vectores numéricos con `TfidfVectorizer`. Como el preprocesamiento (tokenización, stopwords, normalización) ya lo hicimos nosotros, acá solo controlamos:

- `ngram_range`: `(1,1)` solo unigramas, o `(1,2)` unigramas + bigramas (captura algo de contexto, ej. `"new york"`).
- `max_features`: tamaño máximo del vocabulario (se queda con los términos más frecuentes).
- `min_df`: un término tiene que aparecer en al menos `min_df` documentos para entrar al vocabulario (filtra términos rarísimos / errores de tipeo).

### TODO 2: completar `vectorize_data`

In [ ]:
def vectorize_data(train_texts, test_texts, ngram_range, max_features, min_df):
    # El vectorizador aprende el vocabulario y el IDF SOLO con train (fit),
    # y después transforma train y test con ese mismo vocabulario (así no se filtra info del test)
    vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features, min_df=min_df)
    X_train = vectorizer.fit_transform(train_texts)
    X_test = vectorizer.transform(test_texts)
    return X_train, X_test, vectorizer

In [ ]:
# Sanity check rápido con pocos documentos
_sample_train = ["the cat sat on the mat", "dogs are running in the park", "cats and dogs are pets"]
_sample_test = ["the dog sat on the mat"]
_Xtr, _Xte, _vec = vectorize_data(_sample_train, _sample_test, ngram_range=(1, 1), max_features=50, min_df=2)
assert _Xtr is not None, "Completá vectorize_data"
print("Shape train:", _Xtr.shape, "| Shape test:", _Xte.shape)
print("Vocabulario (parcial):", list(_vec.vocabulary_.keys())[:10])


### Qué hace `vectorize_data`

- Creo un `TfidfVectorizer` con los tres parámetros que después busca Optuna (`ngram_range`, `max_features`, `min_df`).
- Con `fit_transform` sobre **train**, el vectorizador aprende el vocabulario (qué términos van a ser columnas) y el IDF de cada término, y transforma esos textos.
- Con `transform` sobre el otro conjunto (validación o test) uso **ese mismo** vocabulario e IDF. No hago `fit` con test porque estaría usando información del test para armar las features (data leakage), y además las columnas no coincidirían con las de train.
- El resultado es una matriz rala (sparse): cada fila es un documento, cada columna un término, y cada valor es TF × IDF, o sea, cuántas veces aparece el término en el documento multiplicado por qué tan raro es el término en el corpus. Una palabra que aparece en casi todos los documentos tiene IDF bajo y pesa poco.
- Por defecto `TfidfVectorizer` normaliza cada **fila** con norma L2 (cada documento queda con largo 1), para que un documento largo no tenga valores más grandes que uno corto solo por tener más palabras.

En el sanity check, con `min_df=2` solo entran al vocabulario las palabras que aparecen en al menos 2 de los 3 documentos ("the", "dogs", "are"), por eso la matriz tiene 3 columnas.

## NOTA: ¿estandarizar las features?

Consideré estandarizar cada feature, o sea, a cada columna restarle su media y dividirla por su desvío. En la primera entrega dije que no hacía falta porque sería una "doble estandarización", pero ese argumento era impreciso, porque TF-IDF no estandariza:
- El **IDF** le da un peso a cada término según qué tan raro es en el corpus, pero no centra la columna ni la lleva a desvío 1.
- La **normalización L2** actúa sobre cada **fila** (cada documento queda con largo 1), no sobre cada **columna**.
- **Estandarizar** actúa sobre cada columna y busca otra cosa: que todas las features tengan media 0 y desvío 1.

Son operaciones sobre ejes distintos y con objetivos distintos. Los motivos reales para no estandarizar son otros:
1. **Restar la media rompe la "raleza"**: la matriz es casi toda ceros (cada documento usa una parte mínima del vocabulario), y al restar la media la ausencia de una palabra pasa a valer un número negativo distinto en cada columna.
2. **Dividir por el desvío infla los términos raros**: una palabra que aparece en muy pocos documentos tiene un desvío muy chico, y al dividir queda con valores enormes, justo en las features que más ruido tienen.
3. **No hay un problema de escala que resolver**: todos los valores ya están entre 0 y 1 por la normalización L2, así que no hay features con valores gigantes que dominen el gradiente.

Para no quedarme solo con el argumento teórico lo probé: en la sección de análisis comparo entrenar sin estandarizar, dividiendo solo por el desvío y con la estandarización completa (ver "¿Conviene estandarizar las features?").

## 4. Modelo: Regresión Softmax

La regresión softmax (regresión logística multinomial) es el modelo más simple posible para clasificación multiclase: **una sola capa lineal** que mapea el vector de entrada a un score (logit) por clase, sin ninguna no-linealidad ni capa oculta.

`CrossEntropyLoss` de PyTorch ya incluye internamente un `log_softmax` + `NLLLoss`, así que el modelo **no debe aplicar softmax en el forward** — hay que devolver los logits crudos.

### TODO 3: completar `SoftmaxRegression`

In [ ]:
class SoftmaxRegression(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        # Una única capa lineal: matriz de pesos W (num_classes x input_dim) + bias b
        self.linear = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        # Devuelvo los logits z = W·x + b (un puntaje por clase), SIN softmax:
        # CrossEntropyLoss ya lo aplica por dentro
        return self.linear(x)

In [ ]:
# Sanity check
_dummy = torch.randn(4, 50)
_model_check = SoftmaxRegression(input_dim=50, num_classes=20)
_out = _model_check(_dummy)
assert _out is not None and _out.shape == (4, 20), "Revisá SoftmaxRegression"
print("Shape de salida OK:", _out.shape)


### Qué hace `SoftmaxRegression`

- En `__init__` defino una sola capa `nn.Linear(input_dim, num_classes)`. Por dentro tiene una matriz de pesos W de tamaño 20 × (cantidad de features) y un bias b de 20 valores: es decir, una fila de pesos por clase.
- En `forward` calculo z = W·x + b, que da 20 números (logits): un puntaje por clase. Cada peso dice cuánto suma o resta una palabra al puntaje de una clase (por ejemplo, la palabra "hockey" debería sumar mucho al puntaje de `rec.sport.hockey`).
- No aplico softmax en el forward porque `CrossEntropyLoss` ya lo hace por dentro (ver respuesta 2). Para predecir alcanza con el `argmax` de los logits, porque softmax no cambia cuál es el más grande.
- Como no hay capas ocultas ni funciones de activación, el modelo es lineal: es exactamente una regresión logística multinomial. Con 20.000 features tiene 20 × 20.000 + 20 = 400.020 parámetros.

## 5. Entrenamiento (ya resuelto)

Dado que los datos vectorizados entran en memoria sin problema, armamos `DataLoader`s a partir de tensores densos. Logueamos loss y accuracy (train y validación) por época a TensorBoard, y a MLflow además los hiperparámetros y el modelo final.

In [ ]:
%load_ext tensorboard


In [ ]:
def to_tensor_dataset(X, y):
    X_dense = torch.tensor(X.toarray(), dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.long)
    return TensorDataset(X_dense, y_tensor)


def train_softmax_model(model, train_loader, val_loader, optimizer, epochs, writer=None, run_name="softmax",
                        restore_best=False, verbose=True):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "epoch_time": []}
    best_val_loss, best_state = float("inf"), None

    for epoch in range(epochs):
        t0 = time.time()
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * xb.size(0)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            total += xb.size(0)
        train_loss, train_acc = running_loss / total, correct / total

        model.eval()
        running_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                logits = model(xb)
                loss = criterion(logits, yb)
                running_loss += loss.item() * xb.size(0)
                correct += (logits.argmax(dim=1) == yb).sum().item()
                total += xb.size(0)
        val_loss, val_acc = running_loss / total, correct / total

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["epoch_time"].append(time.time() - t0)  # segundos de la época (train + validación)

        # Guardo una copia de los pesos de la época con menor loss de validación
        if restore_best and val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())

        if verbose:
            print(f"[{run_name}] Epoch {epoch+1}/{epochs} - "
                  f"train_loss: {train_loss:.4f} train_acc: {train_acc:.4f} - "
                  f"val_loss: {val_loss:.4f} val_acc: {val_acc:.4f}")

        if writer is not None:
            writer.add_scalars("loss", {"train": train_loss, "val": val_loss}, epoch)
            writer.add_scalars("accuracy", {"train": train_acc, "val": val_acc}, epoch)
        mlflow.log_metric("train_loss", train_loss, step=epoch)
        mlflow.log_metric("val_loss", val_loss, step=epoch)
        mlflow.log_metric("train_acc", train_acc, step=epoch)
        mlflow.log_metric("val_acc", val_acc, step=epoch)

    # Al terminar, me quedo con los pesos de la mejor época y no con los de la última
    if restore_best and best_state is not None:
        model.load_state_dict(best_state)

    return history

### Cómo funciona el loop de entrenamiento

`to_tensor_dataset` pasa la matriz rala de TF-IDF a un tensor denso de PyTorch (`float32`) y las etiquetas a enteros (`long`, que es lo que pide `CrossEntropyLoss`), y los junta en un `TensorDataset`. El `DataLoader` después los reparte en mini-batches.

`train_softmax_model` repite esto en cada época:

**Entrenamiento** (`model.train()`). Para cada mini-batch (el `DataLoader` los mezcla en cada época):
1. `optimizer.zero_grad()`: borro los gradientes del batch anterior (si no, PyTorch los va sumando).
2. `logits = model(xb)`: forward, calculo los puntajes de cada clase.
3. `loss = criterion(logits, yb)`: calculo la cross-entropy contra las etiquetas reales.
4. `loss.backward()`: backpropagation, PyTorch calcula el gradiente de la loss respecto de cada peso.
5. `optimizer.step()`: Adam actualiza los pesos en la dirección que baja la loss.
6. Acumulo loss y aciertos para sacar el promedio de la época. Multiplico la loss por `xb.size(0)` porque `CrossEntropyLoss` devuelve el promedio del batch, y el último batch puede ser más chico.

**Validación** (`model.eval()` + `torch.no_grad()`): paso los datos de validación sin calcular gradientes ni tocar los pesos, solo para medir loss y accuracy.

Al final de cada época guardo las métricas en `history`, las imprimo y las mando a TensorBoard y a MLflow.

**Cambios que le agregué al código base:**
- `history["epoch_time"]`: mido cuántos segundos tarda cada época, para analizar los tiempos de convergencia con datos y no a ojo.
- `restore_best`: si es `True`, guardo una copia de los pesos de la época con menor loss de validación y al final los restauro. Así el modelo final es el de la mejor época y no el de la última, que puede estar sobreajustado.
- `verbose`: para apagar los prints en los experimentos que tienen muchas corridas.

Detalle: el accuracy de train que se reporta es el promedio *durante* la época, mientras los pesos van cambiando, así que es un poco más bajo que si lo midiera con los pesos del final de la época.

## 6. Búsqueda de hiperparámetros con Optuna

La búsqueda incluye **hiperparámetros de preprocesamiento** (`remove_stopwords`, `normalization`, `ngram_range`, `max_features`, `min_df`) y **de entrenamiento** (`lr`, `batch_size`), todos en un mismo `objective`.

**Optimización de tiempo**: como la tokenización + stopwords + normalización es lo más lento del pipeline (y solo depende de `remove_stopwords` y `normalization`, que tienen pocas combinaciones posibles), la precalculamos una sola vez para las 6 combinaciones posibles y la cacheamos. Así, cada trial de Optuna solo repite la parte rápida (vectorizar + entrenar pocas épocas).

**Qué hace cada trial** (`objective`): Optuna sugiere un valor para cada hiperparámetro → tomo los textos ya preprocesados de la caché → separo un 20% del train como validación (estratificado y siempre el mismo split, el test no se toca) → vectorizo → entreno 5 épocas → devuelvo el accuracy de validación de la última época, que es lo que Optuna intenta maximizar.

**Cambios respecto de la primera entrega:**
- El sampler TPE tiene semilla (`TPESampler(seed=SEED)`) y cada trial arranca con la misma semilla de PyTorch, así la búsqueda se puede reproducir.
- Cada trial se registra como una corrida propia en MLflow (con sus hiperparámetros). Antes no se abría una corrida por trial y MLflow mezclaba las métricas de todos en una sola.
- Guardo la curva de validación y el tiempo de entrenamiento de cada trial (`set_user_attr`) para analizarlos después.

In [ ]:
print("Precalculando preprocesamiento para las 6 combinaciones posibles (puede tardar unos minutos)...")
preprocessed_cache = {}
for remove_stopwords in [False, True]:
    for normalization in ["none", "stem", "lemmatize"]:
        key = (remove_stopwords, normalization)
        train_proc = [preprocess_text(t, remove_stopwords, normalization) for t in newsgroups_train.data]
        test_proc  = [preprocess_text(t, remove_stopwords, normalization) for t in newsgroups_test.data]
        preprocessed_cache[key] = (train_proc, test_proc)
        print("  listo:", key)


In [ ]:
def objective(trial):
    remove_stopwords = trial.suggest_categorical("remove_stopwords", [False, True])
    normalization = trial.suggest_categorical("normalization", ["none", "stem", "lemmatize"])
    ngram_range = trial.suggest_categorical("ngram_range", ["1gram", "1_2gram"])
    ngram_range_tuple = (1, 1) if ngram_range == "1gram" else (1, 2)
    max_features = trial.suggest_categorical("max_features", [5000, 10000, 20000])
    min_df = trial.suggest_categorical("min_df", [1, 2, 5])
    lr = trial.suggest_float("lr", 1e-3, 1e-1, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])

    set_seed(SEED)  # misma inicialización y mezcla de batches en todos los trials

    train_texts, test_texts = preprocessed_cache[(remove_stopwords, normalization)]
    # Partimos una porción de train como validación para la búsqueda (no usamos el test acá)
    tr_texts, val_texts, tr_labels, val_labels = train_test_split(
        train_texts, newsgroups_train.target, test_size=0.2, random_state=SEED,
        stratify=newsgroups_train.target,
    )

    X_tr, X_val, _ = vectorize_data(tr_texts, val_texts, ngram_range_tuple, max_features, min_df)
    train_loader = DataLoader(to_tensor_dataset(X_tr, tr_labels), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(to_tensor_dataset(X_val, val_labels), batch_size=batch_size, shuffle=False)

    model = SoftmaxRegression(input_dim=X_tr.shape[1], num_classes=NUM_CLASSES)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    # Cada trial es una corrida separada en MLflow, con sus hiperparámetros
    with mlflow.start_run(run_name=f"trial_{trial.number}"):
        mlflow.log_params(trial.params)
        history = train_softmax_model(model, train_loader, val_loader, optimizer, epochs=5,
                                      run_name=f"trial_{trial.number}")

    # Guardo la curva y el tiempo del trial para el análisis
    trial.set_user_attr("val_acc_hist", history["val_acc"])
    trial.set_user_attr("seg_entrenamiento", sum(history["epoch_time"]))
    return history["val_acc"][-1]


mlflow.end_run()  # por si quedó alguna corrida abierta
mlflow.set_experiment("softmax_20newsgroups_busqueda")

N_TRIALS = 40
sampler = optuna.samplers.TPESampler(seed=SEED)  # semilla: la búsqueda es reproducible
study = optuna.create_study(direction="maximize", sampler=sampler)
t_inicio_busqueda = time.time()
study.optimize(objective, n_trials=N_TRIALS)
minutos_busqueda = (time.time() - t_inicio_busqueda) / 60

print("Mejores hiperparámetros:", study.best_params)
print("Mejor accuracy de validación:", study.best_value)
print(f"Tiempo total de la búsqueda: {minutos_busqueda:.1f} minutos")

## NOTA

Aumenté la cantidad de trials para tener más chances de encontrar los mejores parámetros, y también agregué 20.000 como opción de `max_features`, porque en búsquedas anteriores aumentar el vocabulario parecía subir bastante la precisión. Más abajo (sección "¿Hacían falta 40 trials?") analizo con datos si los trials extra sirvieron.

## 7. Entrenamiento final

Uso los mejores hiperparámetros que encontró Optuna.

**Corrección respecto de la primera entrega:** el código base le pasaba el **test** a `train_softmax_model` como si fuera la validación. Entonces el "val_loss" y el "val_acc" que se veían eran en realidad del test, y se usaba el test para seguir el entrenamiento. Ahora:
- Separo el train en 80% entrenamiento / 20% validación (el mismo split que usó la búsqueda).
- Entreno 20 épocas mirando la validación real, y me quedo con los pesos de la época con menor loss de validación (`restore_best=True`).
- El test se usa **una sola vez**, al final, para medir el modelo elegido.

In [ ]:
best = study.best_params
BEST_REMOVE_SW = best["remove_stopwords"]
BEST_NORM = best["normalization"]
BEST_NGRAM = (1, 1) if best["ngram_range"] == "1gram" else (1, 2)
BEST_MAX_FEATURES = best["max_features"]
BEST_MIN_DF = best["min_df"]
BEST_LR = best["lr"]
BEST_BATCH_SIZE = best["batch_size"]
EPOCHS_FINAL = 20

set_seed(SEED)
train_texts_final, test_texts_final = preprocessed_cache[(BEST_REMOVE_SW, BEST_NORM)]

# Validación real: el mismo 20% del train que usó la búsqueda. El test queda aparte.
tr_texts_final, val_texts_final, y_tr_final, y_val_final = train_test_split(
    train_texts_final, newsgroups_train.target, test_size=0.2, random_state=SEED,
    stratify=newsgroups_train.target,
)
X_tr_final, X_val_final, vectorizer_final = vectorize_data(
    tr_texts_final, val_texts_final, BEST_NGRAM, BEST_MAX_FEATURES, BEST_MIN_DF
)
X_test_final = vectorizer_final.transform(test_texts_final)
y_test_final = newsgroups_test.target

train_loader_final = DataLoader(to_tensor_dataset(X_tr_final, y_tr_final),
                                batch_size=BEST_BATCH_SIZE, shuffle=True)
val_loader_final = DataLoader(to_tensor_dataset(X_val_final, y_val_final),
                              batch_size=BEST_BATCH_SIZE, shuffle=False)
test_loader_final = DataLoader(to_tensor_dataset(X_test_final, y_test_final),
                               batch_size=BEST_BATCH_SIZE, shuffle=False)

model = SoftmaxRegression(input_dim=X_tr_final.shape[1], num_classes=NUM_CLASSES)
optimizer = optim.Adam(model.parameters(), lr=BEST_LR)
writer = SummaryWriter(log_dir="runs/softmax_final")

# Nos aseguramos de cerrar cualquier corrida activa antes de configurar la nueva
mlflow.end_run()
mlflow.set_experiment("softmax_20newsgroups")
with mlflow.start_run(run_name="softmax_final"):
    mlflow.log_params({
        "remove_stopwords": BEST_REMOVE_SW, "normalization": BEST_NORM,
        "ngram_range": str(BEST_NGRAM), "max_features": BEST_MAX_FEATURES, "min_df": BEST_MIN_DF,
        "lr": BEST_LR, "batch_size": BEST_BATCH_SIZE, "epochs": EPOCHS_FINAL,
    })
    history = train_softmax_model(model, train_loader_final, val_loader_final, optimizer,
                                  epochs=EPOCHS_FINAL, writer=writer, run_name="final", restore_best=True)
    BEST_EPOCH = int(np.argmin(history["val_loss"])) + 1
    mlflow.log_metric("best_epoch", BEST_EPOCH)
    # Cambiamos el formato de serialización a pickle para evitar el requerimiento de input_example
    mlflow.pytorch.log_model(model, "model", serialization_format="pickle")

writer.close()
print(f"\nMejor época (menor loss de validación): {BEST_EPOCH} -> "
      f"val_loss {history['val_loss'][BEST_EPOCH-1]:.4f}, val_acc {history['val_acc'][BEST_EPOCH-1]:.4f}")
print(f"Tiempo total: {sum(history['epoch_time']):.1f} s ({np.mean(history['epoch_time']):.2f} s por época)")

In [ ]:
# Curvas del entrenamiento final: train vs validación real
ep = np.arange(1, EPOCHS_FINAL + 1)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(ep, history["train_loss"], marker="o", ms=4, label="Train")
axes[0].plot(ep, history["val_loss"], marker="o", ms=4, label="Validación")
axes[0].set_title("Loss por época")
axes[0].set_ylabel("Cross-entropy")

axes[1].plot(ep, history["train_acc"], marker="o", ms=4, label="Train")
axes[1].plot(ep, history["val_acc"], marker="o", ms=4, label="Validación")
axes[1].set_title("Accuracy por época")
axes[1].set_ylabel("Accuracy")

for ax in axes:
    ax.axvline(BEST_EPOCH, color="gray", linestyle="--", label=f"Mejor época ({BEST_EPOCH})")
    ax.set_xlabel("Época")
    ax.set_xticks(ep)
    ax.grid(alpha=0.3)
    ax.legend()
fig.suptitle("Entrenamiento final (validación = 20% separado del train)", fontsize=14)
plt.tight_layout()
plt.show()

print(pd.DataFrame(history, index=ep).round(4).to_string())

### Análisis de las curvas del entrenamiento final

> **[PENDIENTE: se completa con los números de la nueva corrida]**

In [ ]:
%tensorboard --logdir runs


## 8. Evaluación final

In [ ]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for xb, yb in test_loader_final:
        xb = xb.to(device)
        logits = model(xb)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(yb.numpy())

test_accuracy = accuracy_score(all_labels, all_preds)
print(f"Accuracy en test: {test_accuracy:.4f}\n")
print(classification_report(all_labels, all_preds, target_names=class_names))


In [ ]:
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(12, 10))
plt.imshow(cm, cmap="Blues")
plt.colorbar()
plt.xticks(range(NUM_CLASSES), class_names, rotation=90)
plt.yticks(range(NUM_CLASSES), class_names)
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.title("Matriz de confusión - Regresión Softmax sobre 20 Newsgroups")
plt.tight_layout()
plt.show()


### Baselines y análisis de sesgo/varianza

Para saber si el accuracy del modelo es bueno o malo necesito un punto de comparación:
- **Clase más frecuente** (`DummyClassifier`): predice siempre la misma clase. Es el piso mínimo: cualquier modelo que aprenda algo tiene que superarlo.
- **Naive Bayes multinomial**: el clasificador clásico para texto, muy rápido y sin hiperparámetros que buscar. Usa las mismas features TF-IDF, así que compara directamente el modelo y no el preprocesamiento.

Además, para cada modelo comparo el error en train, en validación y en test:
- **Error de train alto** → el modelo no llega a aprender ni los datos que ve (sesgo alto, subajuste).
- **Error de train bajo pero de validación mucho más alto** → aprende de memoria cosas que no generalizan (varianza alta, sobreajuste).

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import MultinomialNB


def evaluar(model, loader):
    # Accuracy y loss con los pesos actuales (sin actualizar nada)
    model.eval()
    criterion = nn.CrossEntropyLoss(reduction="sum")
    total_loss, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            total_loss += criterion(logits, yb).item()
            correct += (logits.argmax(dim=1) == yb).sum().item()
            n += yb.size(0)
    return total_loss / n, correct / n


filas = []
for nombre, clf in [("Baseline: clase más frecuente", DummyClassifier(strategy="most_frequent")),
                    ("Baseline: Naive Bayes multinomial", MultinomialNB())]:
    clf.fit(X_tr_final, y_tr_final)
    filas.append({"modelo": nombre,
                  "acc_train": clf.score(X_tr_final, y_tr_final),
                  "acc_val": clf.score(X_val_final, y_val_final),
                  "acc_test": clf.score(X_test_final, y_test_final)})

fila_softmax = {"modelo": f"Regresión softmax (época {BEST_EPOCH})"}
for col, loader in [("train", train_loader_final), ("val", val_loader_final), ("test", test_loader_final)]:
    loss_, acc_ = evaluar(model, loader)
    fila_softmax[f"acc_{col}"] = acc_
    fila_softmax[f"loss_{col}"] = loss_
filas.append(fila_softmax)

tabla_bv = pd.DataFrame(filas).set_index("modelo")
tabla_bv["error_train"] = 1 - tabla_bv["acc_train"]
tabla_bv["error_val"] = 1 - tabla_bv["acc_val"]
tabla_bv["brecha_val_menos_train"] = tabla_bv["error_val"] - tabla_bv["error_train"]
tabla_bv["brecha_val_menos_test"] = tabla_bv["acc_val"] - tabla_bv["acc_test"]

columnas = ["acc_train", "acc_val", "acc_test", "error_train", "error_val",
            "brecha_val_menos_train", "brecha_val_menos_test", "loss_train", "loss_val", "loss_test"]
display(tabla_bv[columnas].round(4))
print(f"\nMejor valor de la búsqueda de Optuna (val, 5 épocas): {study.best_value:.4f}")

### Conclusiones: baselines, sesgo/varianza y brecha validación-test

> **[PENDIENTE: se completa con los números de la nueva corrida]**

## Análisis de resultados

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Convertimos los resultados de la búsqueda de Optuna a un DataFrame de Pandas para graficar fácilmente
df_results = study.trials_dataframe()

# Renombramos las columnas para que sean más legibles
df_results = df_results.rename(columns={
    "value": "Accuracy de Validación",
    "params_remove_stopwords": "Quitar Stopwords",
    "params_normalization": "Normalización",
    "params_max_features": "Max Features",
    "params_lr": "Learning Rate"
})

# Creamos una figura con subplots separados físicamente
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Gráfico 1: Boxplot de Quitar Stopwords (True vs False de forma discreta)
sns.boxplot(
    data=df_results,
    x="Quitar Stopwords",
    y="Accuracy de Validación",
    ax=axes[0],
    palette="Set2",
    hue="Quitar Stopwords",
    legend=False
)
sns.stripplot(data=df_results, x="Quitar Stopwords", y="Accuracy de Validación", ax=axes[0], color="black", alpha=0.5, size=6)
axes[0].set_title("Rendimiento según Quitar Stopwords\n(Separado discretamente sin degradados)")
axes[0].grid(axis='y', linestyle='--', alpha=0.7)

# Gráfico 2: Boxplot de Normalización (none vs stem vs lemmatize)
sns.boxplot(
    data=df_results,
    x="Normalización",
    y="Accuracy de Validación",
    ax=axes[1],
    palette="Set3",
    hue="Normalización",
    legend=False
)
sns.stripplot(data=df_results, x="Normalización", y="Accuracy de Validación", ax=axes[1], color="black", alpha=0.5, size=6)
axes[1].set_title("Rendimiento según Método de Normalización\n(Cada categoría en su columna propia)")
axes[1].grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

### NOTA

En el primer grafico intente entender cuales de los hiperparametros con salidas discretas, tenian los accuracy mas alto. Posteriormente , utilizando esos hice un analisis mas en profundidad con el resto de parametros continuos.

In [ ]:
import itertools
import numpy as np
from scipy.interpolate import griddata

# Filtramos los resultados de Optuna a solo las mejores combinaciones de preprocesamiento
filtered_df = df_results[
    (df_results["Quitar Stopwords"] == BEST_REMOVE_SW) &
    (df_results["Normalización"] == BEST_NORM)
].copy()

params = [
    ("Max Features", "Max Features"),
    ("params_min_df", "Min DF"),
    ("Learning Rate", "Learning Rate"),
    ("params_batch_size", "Batch Size"),
]
log_params = {"Learning Rate"}  # se interpola y grafica en escala logarítmica

acc_col = "Accuracy de Validación"
vmin, vmax = filtered_df[acc_col].min(), filtered_df[acc_col].max()
levels = np.linspace(vmin, vmax, 21)  # misma escala de color en todos los gráficos
best = filtered_df.loc[filtered_df[acc_col].idxmax()]

pairs = list(itertools.combinations(params, 2))
fig, axes = plt.subplots(2, 3, figsize=(18, 10), constrained_layout=True)

for ax, ((x_col, x_name), (y_col, y_name)) in zip(axes.flatten(), pairs):
    # Si hay trials con el mismo (x, y), promediamos su accuracy
    pts = filtered_df.groupby([x_col, y_col], as_index=False)[acc_col].mean()
    x = pts[x_col].astype(float).values
    y = pts[y_col].astype(float).values
    z = pts[acc_col].values

    # Pasamos a log los parámetros que lo necesitan y normalizamos a [0, 1]
    # para que un eje con valores grandes (Max Features) no domine la interpolación
    xt = np.log10(x) if x_col in log_params else x
    yt = np.log10(y) if y_col in log_params else y
    xn = (xt - xt.min()) / (xt.max() - xt.min())
    yn = (yt - yt.min()) / (yt.max() - yt.min())

    # Grilla fina donde estimamos la accuracy
    gxn, gyn = np.meshgrid(np.linspace(0, 1, 200), np.linspace(0, 1, 200))
    gz = griddata((xn, yn), z, (gxn, gyn), method="linear")
    # Afuera de la zona cubierta por los puntos, usamos el trial más cercano
    gz_near = griddata((xn, yn), z, (gxn, gyn), method="nearest")
    gz = np.where(np.isnan(gz), gz_near, gz)

    # Volvemos la grilla a las unidades originales para graficar
    gx = xt.min() + gxn * (xt.max() - xt.min())
    gy = yt.min() + gyn * (yt.max() - yt.min())
    if x_col in log_params:
        gx = 10 ** gx
    if y_col in log_params:
        gy = 10 ** gy

    cf = ax.contourf(gx, gy, gz, levels=levels, cmap="viridis", extend="both")

    # Trials reales encima
    ax.scatter(x, y, c="white", s=25, edgecolors="black", linewidths=0.6, label="Trials reales")
    ax.scatter(best[x_col], best[y_col], marker="*", s=450, c="red",
               edgecolors="black", linewidths=1, label=f"Mejor trial ({best[acc_col]:.4f})")

    if x_col in log_params:
        ax.set_xscale("log")
    if y_col in log_params:
        ax.set_yscale("log")
    ax.set_xlabel(x_name)
    ax.set_ylabel(y_name)
    ax.set_title(f"{y_name} vs {x_name}")

axes.flatten()[0].legend(loc="best")
fig.colorbar(cf, ax=axes, label=acc_col, shrink=0.8)
fig.suptitle(
    f"Mapa de accuracy interpolado (Stopwords={BEST_REMOVE_SW}, Norm={BEST_NORM})",
    fontsize=15
)
plt.show()



### Evidencia: resultados de la búsqueda por hiperparámetro

Para respaldar las conclusiones con datos, agrupo los 40 trials por cada hiperparámetro y miro cuántos trials hubo con cada valor, el accuracy promedio, el desvío y el máximo. El learning rate, al ser continuo, lo agrupo en rangos.

Ojo con cómo se lee esta tabla: los trials no varían un hiperparámetro por vez, así que la diferencia entre dos valores puede deberse a los otros hiperparámetros que los acompañaban (por ejemplo, si TPE probó `batch_size=128` sobre todo con learning rates malos). Por eso después hago experimentos controlados, variando un solo hiperparámetro y con varias semillas.

In [ ]:
df_trials = study.trials_dataframe()
df_trials["duracion_seg"] = df_trials["duration"].dt.total_seconds()

def resumen_por(columna, etiqueta):
    t = df_trials.groupby(columna, observed=True)["value"].agg(
        trials="count", acc_media="mean", acc_desvio="std", acc_max="max")
    t.index = [f"{etiqueta} = {v}" for v in t.index]
    return t

tablas = [resumen_por(f"params_{p}", p) for p in
          ["remove_stopwords", "normalization", "ngram_range", "max_features", "min_df", "batch_size"]]
rangos_lr = pd.cut(df_trials["params_lr"], bins=[1e-3, 3e-3, 1e-2, 3e-2, 1e-1], include_lowest=True,
                   labels=["0.001-0.003", "0.003-0.01", "0.01-0.03", "0.03-0.1"])
tablas.append(resumen_por(rangos_lr, "lr en"))
tabla_hp = pd.concat(tablas)
display(tabla_hp.round(4))

# Las 10 mejores configuraciones
cols_top = ["number", "value"] + [c for c in df_trials.columns if c.startswith("params_")] + ["duracion_seg"]
display(df_trials.sort_values("value", ascending=False)[cols_top].head(10).round(4))

### Importancia de los hiperparámetros

Optuna puede estimar qué hiperparámetros explican más la variación del accuracy entre trials (método fANOVA: ajusta un random forest que predice el accuracy a partir de los hiperparámetros y mide cuánta varianza explica cada uno). Los valores suman 1. Como son solo 40 trials, es una estimación aproximada y hay que leerla junto con la tabla de arriba.

In [ ]:
importancias = optuna.importance.get_param_importances(
    study, evaluator=optuna.importance.FanovaImportanceEvaluator(seed=SEED))
serie_imp = pd.Series(importancias).sort_values()

fig, ax = plt.subplots(figsize=(9, 4.5))
serie_imp.plot.barh(ax=ax, color="#2a78d6")
for i, v in enumerate(serie_imp.values):
    ax.text(v + 0.005, i, f"{v:.2f}", va="center")
ax.set_xlabel("Importancia (fANOVA)")
ax.set_title("Importancia de cada hiperparámetro en el accuracy de validación")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

### ¿Hacían falta 40 trials?

En la primera entrega afirmé que con 40 trials se obtuvo un mejor resultado que con 15, pero no tenía una corrida de 15 trials que lo respalde. Una forma de verlo con los datos de esta búsqueda es la curva del "mejor resultado hasta ahora": un estudio de 15 trials con la misma semilla habría probado exactamente los primeros 15 trials de este, así que el mejor valor a los 15 trials es lo que habría encontrado. También marco el trial 10, porque hasta ahí TPE elige al azar (son sus trials de arranque) y recién después empieza a usar lo aprendido.

In [ ]:
valores = df_trials.sort_values("number")["value"].values
mejor_hasta_ahora = np.maximum.accumulate(valores)
trials_eje = np.arange(1, len(valores) + 1)

fig, ax = plt.subplots(figsize=(11, 5))
ax.scatter(trials_eje, valores, color="#999999", s=25, label="Accuracy de cada trial")
ax.step(trials_eje, mejor_hasta_ahora, where="post", color="#eb6834", linewidth=2, label="Mejor hasta ese trial")
ax.axvline(10, color="#2a78d6", linestyle=":", label="Fin de los trials aleatorios de TPE (10)")
ax.axvline(15, color="black", linestyle="--", label="15 trials")
ax.set_xlabel("Número de trial")
ax.set_ylabel("Accuracy de validación")
ax.set_title("Historia de la optimización")
ax.grid(alpha=0.3)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

print(f"Mejor accuracy con 15 trials: {mejor_hasta_ahora[14]:.4f}")
print(f"Mejor accuracy con {len(valores)} trials: {mejor_hasta_ahora[-1]:.4f}")
print(f"Diferencia: {mejor_hasta_ahora[-1] - mejor_hasta_ahora[14]:.4f}")
print(f"Trial donde apareció el mejor: {int(np.argmax(valores)) + 1}")
print(f"Accuracy promedio de los trials 1-10 (aleatorios): {valores[:10].mean():.4f} | "
      f"de los trials 11-{len(valores)} (TPE): {valores[10:].mean():.4f}")

### Conclusión de la búsqueda de hiperparámetros

> **[PENDIENTE: se completa con los números de la nueva corrida]**

### Experimentos controlados: un hiperparámetro por vez y con varias semillas

En la primera entrega analicé la dinámica del entrenamiento solo con los logs de 5 épocas de 3 trials de la búsqueda. Esos trials cambiaban varios hiperparámetros a la vez y tenían una sola corrida, así que no se podía saber si una diferencia se debía al learning rate, a otro hiperparámetro o simplemente al azar.

Ahora hago experimentos controlados:
- Dejo fijo el preprocesamiento y el vocabulario del entrenamiento final (los mejores de Optuna) y el mismo split de validación.
- **Barrido de learning rate**: pruebo `lr = 0.001, 0.01, 0.1` (los extremos y el medio del rango de la búsqueda, en escala log) con el mejor batch size.
- **Barrido de batch size**: pruebo `32, 64, 128` con el mejor learning rate.
- Cada configuración la corro con **3 semillas** distintas (cambia la inicialización de los pesos y el orden de los batches). Grafico el promedio y una banda de ± 1 desvío, que muestra cuánto ruido hay entre corridas.
- Entreno 15 épocas para ver más allá de las 5 de la búsqueda, y mido el tiempo de cada época.

Son 18 corridas en total; en CPU tarda alrededor de 10 minutos.

In [ ]:
EPOCHS_EXP = 15
SEMILLAS = [0, 1, 2]
LRS = [0.001, 0.01, 0.1]
BATCH_SIZES = [32, 64, 128]

# Reuso los mismos tensores para todas las corridas (vectorizar una sola vez)
train_ds_exp = train_loader_final.dataset
val_ds_exp = val_loader_final.dataset

mlflow.end_run()
mlflow.set_experiment("softmax_20newsgroups_controlados")
resultados_exp = {}


def correr_config(lr, batch_size, seed, epochs=EPOCHS_EXP, train_ds=None, val_ds=None, nombre=""):
    train_ds = train_ds if train_ds is not None else train_ds_exp
    val_ds = val_ds if val_ds is not None else val_ds_exp
    set_seed(seed)
    tr_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    va_loader = DataLoader(val_ds, batch_size=256, shuffle=False)  # el batch de validación no afecta el aprendizaje
    m = SoftmaxRegression(input_dim=train_ds.tensors[0].shape[1], num_classes=NUM_CLASSES)
    opt = optim.Adam(m.parameters(), lr=lr)
    run_name = f"{nombre}lr={lr:.4g}_bs={batch_size}_seed={seed}"
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({"lr": lr, "batch_size": batch_size, "seed": seed, "epochs": epochs})
        return train_softmax_model(m, tr_loader, va_loader, opt, epochs, run_name=run_name, verbose=False)


configs_lr = [(lr, BEST_BATCH_SIZE) for lr in LRS]
configs_bs = [(BEST_LR, bs) for bs in BATCH_SIZES]
t0 = time.time()
for lr, bs in configs_lr + configs_bs:
    for s in SEMILLAS:
        if (lr, bs, s) not in resultados_exp:
            resultados_exp[(lr, bs, s)] = correr_config(lr, bs, s)
    print(f"lr={lr:.4g}, batch_size={bs}: listo ({(time.time() - t0) / 60:.1f} min acumulados)")

In [ ]:
def graficar_barrido(configs, etiquetas, titulo):
    metricas = [("train_acc", "Accuracy train"), ("val_acc", "Accuracy validación"), ("val_loss", "Loss validación")]
    ep = np.arange(1, EPOCHS_EXP + 1)
    fig, axes = plt.subplots(1, 3, figsize=(19, 5))
    for (lr, bs), etiqueta in zip(configs, etiquetas):
        for ax, (m, _) in zip(axes, metricas):
            curvas = np.array([resultados_exp[(lr, bs, s)][m] for s in SEMILLAS])
            media, desvio = curvas.mean(axis=0), curvas.std(axis=0)
            linea, = ax.plot(ep, media, marker="o", ms=3, label=etiqueta)
            ax.fill_between(ep, media - desvio, media + desvio, color=linea.get_color(), alpha=0.2)
    for ax, (_, nombre) in zip(axes, metricas):
        ax.set_title(nombre)
        ax.set_xlabel("Época")
        ax.grid(alpha=0.3)
    axes[0].legend()
    fig.suptitle(titulo + f" (promedio ± desvío de {len(SEMILLAS)} semillas)", fontsize=14)
    plt.tight_layout()
    plt.show()


graficar_barrido(configs_lr, [f"lr = {lr}" for lr in LRS],
                 f"Barrido de learning rate (batch size = {BEST_BATCH_SIZE})")
graficar_barrido(configs_bs, [f"batch size = {bs}" for bs in BATCH_SIZES],
                 f"Barrido de batch size (lr = {BEST_LR:.4f})")

#### Tabla de convergencia

Para cada configuración (promedio de las 3 semillas) mido:
- `mejor_val_acc` (media y desvío entre semillas) y la época en la que se alcanza.
- `val_acc_final`: el accuracy de validación en la última época (si es menor que el mejor, el modelo empeoró al seguir entrenando).
- `brecha_final`: accuracy de train menos accuracy de validación en la última época (cuánto sobreajusta).
- `seg_por_época`: tiempo promedio de una época.
- `época_umbral` y `seg_umbral`: época y segundos acumulados hasta llegar por primera vez a un accuracy de validación de 72%. `semillas_que_llegan` cuenta en cuántas semillas se llegó (si no llegó, no se promedia).

In [ ]:
UMBRAL_ACC = 0.72

filas = []
for (lr, bs) in configs_lr + configs_bs:
    for s in SEMILLAS:
        h = resultados_exp[(lr, bs, s)]
        va = np.array(h["val_acc"])
        t_acum = np.cumsum(h["epoch_time"])
        llega = np.where(va >= UMBRAL_ACC)[0]
        filas.append({
            "lr": round(lr, 6), "batch_size": bs, "semilla": s,
            "mejor_val_acc": va.max(), "época_mejor": va.argmax() + 1, "val_acc_final": va[-1],
            "brecha_final": h["train_acc"][-1] - va[-1],
            "seg_por_época": np.mean(h["epoch_time"]),
            "época_umbral": llega[0] + 1 if len(llega) else np.nan,
            "seg_umbral": t_acum[llega[0]] if len(llega) else np.nan,
        })
df_exp = pd.DataFrame(filas)

tabla_conv = df_exp.groupby(["lr", "batch_size"]).agg(
    mejor_val_acc=("mejor_val_acc", "mean"),
    desvio_mejor_val_acc=("mejor_val_acc", "std"),
    época_mejor=("época_mejor", "mean"),
    val_acc_final=("val_acc_final", "mean"),
    brecha_final=("brecha_final", "mean"),
    seg_por_época=("seg_por_época", "mean"),
    época_umbral=("época_umbral", "mean"),
    seg_umbral=("seg_umbral", "mean"),
    semillas_que_llegan=("época_umbral", "count"),
)
print(f"Umbral de accuracy de validación: {UMBRAL_ACC:.0%}")
display(tabla_conv.round(4))

### Análisis de la dinámica de entrenamiento

> **[PENDIENTE: se completa con los números de la nueva corrida]**

### Tiempos de la búsqueda

Optuna guarda cuándo empieza y termina cada trial, así que puedo medir cuánto tardó cada uno (vectorizar + armar los tensores + entrenar 5 épocas) y ver qué hiperparámetros lo hacen más lento. Igual que antes, en la búsqueda los hiperparámetros cambian todos juntos; el efecto aislado del batch size en el tiempo por época está en la tabla de convergencia de arriba.

In [ ]:
seg_por_trial = {t.number: t.user_attrs.get("seg_entrenamiento", np.nan) for t in study.trials}
df_trials["seg_entrenamiento"] = df_trials["number"].map(seg_por_trial)
print(f"Tiempo total de la búsqueda: {df_trials['duracion_seg'].sum() / 60:.1f} min "
      f"({N_TRIALS} trials, {df_trials['duracion_seg'].mean():.1f} s por trial en promedio)\n")

for col in ["params_ngram_range", "params_max_features", "params_batch_size"]:
    t = df_trials.groupby(col)[["duracion_seg", "seg_entrenamiento"]].agg(["count", "mean"]).round(1)
    display(t)

display(df_trials.groupby(["params_ngram_range", "params_max_features"])["duracion_seg"]
        .agg(["count", "mean", "max"]).round(1))

### Tiempos de convergencia del entrenamiento

> **[PENDIENTE: se completa con los números de la nueva corrida]**

### ¿Conviene estandarizar las features?

Prueba de lo que discutí en la nota de la sección 3. Con los mejores hiperparámetros y el mismo presupuesto que la búsqueda (5 épocas), comparo tres variantes, con 3 semillas cada una:
1. **TF-IDF tal cual** (lo que usé en todo el TP).
2. **Solo dividir por el desvío** de cada feature (sin restar la media, mantiene los ceros).
3. **Estandarización completa**: (x − media) / desvío.

La media y el desvío se calculan **solo con el train** y se aplican igual a la validación, para no usar información de validación.

In [ ]:
EPOCHS_STD = 5
Xtr_base, ytr_base = train_ds_exp.tensors
Xval_base, yval_base = val_ds_exp.tensors
media_tr = Xtr_base.mean(dim=0)
desvio_tr = Xtr_base.std(dim=0) + 1e-8

variantes = {
    "TF-IDF tal cual": lambda X: X,
    "Solo dividir por el desvío": lambda X: X / desvio_tr,
    "Estandarización completa": lambda X: (X - media_tr) / desvio_tr,
}

filas = []
for nombre, transformar in variantes.items():
    ds_tr = TensorDataset(transformar(Xtr_base), ytr_base)
    ds_val = TensorDataset(transformar(Xval_base), yval_base)
    for s in SEMILLAS:
        h = correr_config(BEST_LR, BEST_BATCH_SIZE, s, epochs=EPOCHS_STD,
                          train_ds=ds_tr, val_ds=ds_val, nombre=f"std[{nombre}]_")
        filas.append({"variante": nombre, "semilla": s, "train_acc": h["train_acc"][-1],
                      "val_acc": h["val_acc"][-1], "val_loss": h["val_loss"][-1]})
    del ds_tr, ds_val  # libero memoria

df_std = pd.DataFrame(filas)
tabla_std = df_std.groupby("variante", sort=False).agg(
    val_acc_media=("val_acc", "mean"), val_acc_desvio=("val_acc", "std"),
    train_acc_media=("train_acc", "mean"), val_loss_media=("val_loss", "mean"))
print(f"Rango de valores de las features: TF-IDF [{Xtr_base.min():.2f}, {Xtr_base.max():.2f}] | "
      f"divididas por el desvío: hasta {(Xtr_base / desvio_tr).max():.1f}")
display(tabla_std.round(4))

#### Conclusión sobre la estandarización

> **[PENDIENTE: se completa con los números de la nueva corrida]**

## 9. Preguntas de reflexión

**Sobre Regresión Softmax (teoría)**
1. ¿Por qué la regresión softmax es una generalización de la regresión logística binaria? ¿Qué pasaría si `NUM_CLASSES = 2`?
2. ¿Por qué `CrossEntropyLoss` en PyTorch no necesita que se le aplique `softmax` antes en el forward? ¿Qué pasaría si igual lo aplicaran (softmax + CrossEntropyLoss)?
3. ¿Qué tipo de frontera de decisión puede aprender un modelo de regresión softmax? ¿Qué limitación tiene frente a un problema que no es linealmente separable?
4. ¿En qué se diferencia este modelo de una red neuronal con capas ocultas? ¿En qué situación elegirían una sobre la otra?

**Sobre frameworks y herramientas**
5. ¿Qué ventajas y desventajas tiene implementar esto en PyTorch (como hicimos) comparado con usar directamente `sklearn.linear_model.LogisticRegression(multi_class="multinomial")`?
6. ¿Para qué sirvió específicamente MLflow en este TP? ¿En qué se diferencia de lo que loguea TensorBoard?
7. ¿Qué ventaja concreta les dio Optuna frente a probar combinaciones de hiperparámetros a mano?

**Sobre el experimento**
8. ¿Qué combinación de preprocesamiento (stopwords, stemming/lematización, n-gramas) encontró Optuna como mejor? ¿Tiene sentido para este dataset?
9. ¿En qué categorías del dataset se confunde más el modelo (según la matriz de confusión)? ¿Por qué creen que pasa (piensen en el contenido temático de esas categorías)?
10. ¿Cambiaría su respuesta a la pregunta anterior si usaran un modelo con capas ocultas en vez de regresión softmax? ¿Por qué sí o por qué no?


## Respuestas

1. Como softmax permite obtener una probabilidad de varios tipos de clase, cuando solo hay 2 clases termina teniendo el mismo resultado que una logica binaria, solo que siendo menos efectivo , ya que en ese caso al saber la probabilidad de una clase ya sabes la otra.

2. `CrossEntropyLoss` recibe los logits crudos porque por dentro ya aplica `log_softmax` y después `NLLLoss`. Para un ejemplo de clase real *y* calcula:

   $$\text{loss} = -\log\big(\text{softmax}(z)_y\big) = -z_y + \log \sum_k e^{z_k}$$

   Hacer softmax y logaritmo juntos tiene dos ventajas. Primero, es numéricamente estable: usa el truco log-sum-exp (le resta el logit máximo antes de hacer la exponencial), así no hay overflow con e^z ni se calcula log(0). Segundo, el gradiente respecto de cada logit queda muy simple: $\partial \text{loss} / \partial z_k = p_k - \mathbb{1}[k = y]$, o sea, "probabilidad predicha menos probabilidad real". Si el modelo le da 0,9 a la clase correcta el gradiente es chico, y si le da 0,1 es grande: corrige en proporción a cuánto se equivoca.

   Si además aplicara softmax en el forward, la loss calcularía softmax(softmax(z)). Las entradas del segundo softmax son probabilidades entre 0 y 1, así que su salida queda casi uniforme: en el mejor caso (el modelo le da probabilidad 1 a la clase correcta), el segundo softmax le da $e^1 / (e^1 + 19 \cdot e^0) \approx 0{,}125$ con 20 clases. Eso trae dos problemas:
   - La loss nunca puede bajar de −log(0,125) ≈ 2,08, aunque el modelo acierte todo con total seguridad, así que el modelo sigue recibiendo error aunque ya haya aprendido.
   - El gradiente tiene que pasar además por la derivada del primer softmax ($p_k (1 - p_k)$), que se hace casi 0 cuando las probabilidades se acercan a 0 o a 1. Los gradientes quedan muy chicos y el modelo aprende mucho más lento o se estanca.

   Las predicciones (el `argmax`) no cambiarían, porque softmax no cambia el orden de los valores; lo que se arruina es el entrenamiento.

3. Un modelo softmax solo tiene fronteras de decision lineales, por lo que para un problema muy complejo que requiera de una frontera tipo curva este no va a ser preciso

4. Una red neuronal de capas ocultas, a diferencia de este modelo , permite tener fronteras curvas, por lo que es mejor utilizarlo para problemas mas complejos o con muchos datos, mientras que el softmax es mejor para más simples y tambien mas rapido

5. **Ventajas de PyTorch:**
   - Control total del entrenamiento: elijo el optimizador (Adam), el learning rate, el batch size y las épocas, y veo la loss época por época en TensorBoard y MLflow.
   - Puedo meterme en el loop: guardar los pesos de la mejor época, medir tiempos, etc.
   - Corre en GPU, y el mismo código se extiende fácil a una red con capas ocultas agregando capas al `nn.Module` (que es lo que se hace en el TP2).

   **Desventajas:**
   - Mucho más código y más lugares para equivocarse (por ejemplo, el código base usaba el test como validación).
   - Tuve que pasar la matriz TF-IDF a densa: con 20.000 features son unos 9.000 × 20.000 floats (≈ 720 MB), mientras que `LogisticRegression` trabaja directo con la matriz rala.
   - `LogisticRegression` trae regularización L2 por defecto (`C=1.0`), que ayuda contra el sobreajuste. En mi versión de PyTorch no hay regularización (`weight_decay=0`) y se nota en la brecha entre train y validación.
   - sklearn optimiza con solvers como `lbfgs`, que usan todo el dataset y paran solos cuando convergen: no hay que elegir learning rate, batch size ni épocas, así que hay muchos menos hiperparámetros para buscar.

   (En versiones nuevas de sklearn el parámetro `multi_class` quedó deprecado porque el modo multinomial ya es el comportamiento por defecto.)

6. MLflow lo usé como registro de experimentos. Cada trial de Optuna, cada corrida de los experimentos controlados y el entrenamiento final quedan guardados como una corrida ("run") separada, con sus hiperparámetros (`log_params`), las métricas de cada época (`log_metric`) y, en el caso del final, el modelo entrenado como artefacto (`log_model`), que después se puede volver a cargar sin reentrenar. Desde la interfaz (`mlflow ui`) se pueden filtrar y ordenar las corridas por hiperparámetro o por métrica y comparar varias a la vez.

   TensorBoard, en cambio, está pensado para visualizar: muestra las curvas de loss y accuracy, incluso en tiempo real, y es cómodo para mirar la dinámica de un entrenamiento. Pero no guarda de forma ordenada qué hiperparámetros tenía cada corrida ni el modelo.

   En resumen: TensorBoard sirve para *mirar* cómo entrena un modelo, y MLflow para *guardar y comparar* muchos experimentos y poder reproducirlos. (En la primera entrega los trials no abrían una corrida propia, así que MLflow juntaba las métricas de todos en una sola y no servía para compararlos; ahora cada trial tiene la suya.)

7. Concretamente:
   - El espacio de búsqueda tiene 2 × 3 × 2 × 3 × 3 × 3 = 324 combinaciones de los hiperparámetros discretos, y además el learning rate es continuo. Probar todo a mano (o con una grilla) no era posible en un tiempo razonable; con Optuna alcanzaron 40 trials.
   - TPE no prueba al azar: los primeros 10 trials sí son aleatorios, pero después separa los trials en "buenos" y "malos", arma una distribución de probabilidad para cada grupo y propone valores que se parecen a los buenos y no a los malos. [PENDIENTE: dato de la tabla por hiperparámetro sobre cuántos trials le dedicó a la mejor combinación.]
   - Maneja juntos parámetros categóricos y continuos, y busca el learning rate en escala logarítmica (`log=True`), que es la escala en la que tiene sentido variarlo.
   - Guarda todos los trials con sus resultados, lo que me permitió armar después las tablas, la importancia de hiperparámetros y la historia de la optimización sin anotar nada a mano.

8. > **[PENDIENTE: se completa con los números de la nueva corrida]**

   (Texto de la primera entrega, a actualizar:) Optuna encontró como mejor combinación quitar las stopwords, usar stemming y usar solo unigramas (junto con 20000 features y min_df = 1), llegando a un 75% de accuracy en validación. Tiene sentido para este dataset: las stopwords no ayudan a saber de qué tema habla un texto, y al quitarlas quedan más lugares del vocabulario para palabras útiles. El stemming junta distintas formas de una misma palabra, por lo que con la misma cantidad de features se cubren más conceptos. En cambio, la lematización muchas veces no modifica las palabras. Por último, los bigramas agregan muchas combinaciones que aparecen muy pocas veces, y con la cantidad de datos que tenemos el modelo no llega a aprenderlas bien.

9. > **[PENDIENTE: se completa con los números de la nueva corrida]**

   (Texto de la primera entrega, a actualizar:) El modelo se confunde más entre categorías que tratan temas parecidos. Las peores son talk.religion.misc (acierta solo el 33%), que se confunde con soc.religion.christian y alt.atheism, y talk.politics.misc (43%), que se confunde sobre todo con talk.politics.guns. También hay confusión entre los grupos de computación (por ejemplo, hardware de PC con hardware de Mac, y comp.graphics con comp.windows.x) y entre rec.autos y rec.motorcycles. Esto pasa porque esas categorías usan casi las mismas palabras: en las tres de religión se habla de Dios, la Biblia y la fe, en las de política se discuten temas similares, en las de computación se usan términos técnicos comunes, al igual que en los grupos de autos y motos. Como el modelo decide solo según las palabras que aparecen, si dos temas comparten vocabulario no tiene forma de diferenciarlos.

10. Cambiaría pero no en una gran medida. Una red con capas ocultas podría combinar palabras y encontrar patrones más complejos, lo que podría mejorar algo la precisión. Pero el problema de fondo seguiría, ya que esas categorías realmente hablan de lo mismo, e incluso algunos textos podrían pertenecer a cualquiera de las dos.